# RAPORT Z REALIZACJI PROJEKTU

## Przewidywanie zużycia paliwa w pojazdach
---

**Przedmiot:** Uczenie Maszynowe  
**Zbiór danych:** UCI Auto MPG Dataset  
**Środowisko:** Google Colab (GPU)  
**Biblioteki:** scikit-learn, TensorFlow, pandas, matplotlib, seaborn  

---

## Spis treści

1. [Wstęp](#1-wstęp)
2. [Charakterystyka zbioru danych](#2-charakterystyka-zbioru-danych)
3. [Opis zastosowanych algorytmów uczenia maszynowego](#3-opis-zastosowanych-algorytmów-uczenia-maszynowego)
4. [Metodologia](#4-metodologia)
5. [Optymalizacja hiperparametrów](#5-optymalizacja-hiperparametrów)
6. [Wyniki i analiza](#6-wyniki-i-analiza)
7. [Wnioski](#7-wnioski)

## 1. Wstęp

### 1.1 Cel projektu

Celem niniejszego projektu jest opracowanie i porównanie modeli regresyjnych służących do przewidywania zużycia paliwa (wyrażonego w milach na galon -- MPG) na podstawie cech technicznych pojazdów. Analiza obejmuje pięć algorytmów uczenia maszynowego:

- **MLPRegressor** -- wielowarstwowy perceptron (scikit-learn)
- **RandomForestRegressor** -- las losowy (scikit-learn)
- **SVR** -- regresja wektorów wspierających (scikit-learn)
- **KNeighborsRegressor** -- regresja najbliższych sąsiadów (scikit-learn)
- **Deep Neural Network** -- głęboka sieć neuronowa (TensorFlow/Keras, GPU)

### 1.2 Zakres pracy

Projekt obejmuje następujące etapy:

1. Eksploracyjną analizę danych (EDA)
2. Przetwarzanie wstępne i inżynierię cech
3. Optymalizację hiperparametrów metodą GridSearchCV
4. Trenowanie i walidację modeli
5. Analizę błędów względnych
6. Ocenę dokładności w zależności od typu pojazdu
7. Wizualizacje porównawcze wyników

## 2. Charakterystyka zbioru danych

### 2.1 Źródło i ogólna charakterystyka

Wykorzystany w projekcie zbiór danych *UCI Auto MPG Dataset* pochodzi z repozytorium Carnegie Mellon University i jest dostępny poprzez bibliotekę seaborn. Zbiór zawiera 398 rekordów opisujących pojazdy wyprodukowane w latach 1970--1982.

### 2.2 Struktura atrybutów

| Atrybut | Opis | Typ | Jednostka |
|---------|------|-----|-----------|
| mpg | Zużycie paliwa | Ciągły | mile/galon |
| cylinders | Liczba cylindrów | Dyskretny | szt. |
| displacement | Pojemność silnika | Ciągły | cal³ |
| horsepower | Moc silnika | Ciągły | KM |
| weight | Masa pojazdu | Ciągły | funty |
| acceleration | Przyspieszenie 0-60 mph | Ciągły | sekundy |
| model_year | Rok modelu | Dyskretny | rok |
| origin | Kraj pochodzenia | Kategoryczny | 1/2/3 |
| name | Nazwa pojazdu | Tekstowy | -- |

### 2.3 Pierwsze pięć przykładowych rekordów

| name | mpg | cyl | disp | hp | weight | acc | year | orig |
|------|-----|-----|------|----|----|-----|------|------|
| chevrolet chevelle malibu | 18.0 | 8 | 307.0 | 130 | 3504 | 12.0 | 70 | 1 |
| buick skylark 320 | 15.0 | 8 | 350.0 | 165 | 3693 | 11.5 | 70 | 1 |
| plymouth satellite | 18.0 | 8 | 318.0 | 150 | 3436 | 11.0 | 70 | 1 |
| amc rebel sst | 16.0 | 8 | 304.0 | 150 | 3433 | 12.0 | 70 | 1 |
| ford torino | 17.0 | 8 | 302.0 | 140 | 3449 | 10.5 | 70 | 1 |

### 2.4 Rozkłady cech

Analiza rozkładów poszczególnych cech numerycznych ujawnia następujące właściwości zbioru:

- **mpg:** Rozkład prawostronnie skośny, zakres 9--46.6 MPG, średnia ok. 23.5 MPG
- **cylinders:** Rozkład dyskretny z dominacją 4 i 8 cylindrów
- **displacement:** Rozkład dwumodalny odzwierciedlający podział na silniki 4- i 8-cylindrowe
- **horsepower:** Zakres 46--230 KM, średnia ok. 104 KM
- **weight:** Zakres 1613--5140 funtów, średnia ok. 2978 funtów
- **acceleration:** Rozkład zbliżony do normalnego, średnia ok. 15.5 s
- **model_year:** Rozkład jednostajny w zakresie 70--82

### 2.5 Macierz korelacji

Analiza korelacji liniowej (współczynnik Pearsona) wskazuje na następujące silne zależności ze zmienną docelową *mpg*:

| Cecha | Korelacja z mpg |
|-------|----------------|
| weight | $-0.832$ |
| cylinders | $-0.778$ |
| displacement | $-0.778$ |
| horsepower | $-0.778$ |
| acceleration | $+0.423$ |
| model_year | $+0.581$ |

Najsilniejszą ujemną korelację obserwuje się dla masy pojazdu ($r = -0.832$), co jest zgodne z oczekiwaniami fizycznymi -- cięższe pojazdy zużywają więcej paliwa.

### 2.6 Podział według kraju pochodzenia

| Kraj | Kod | Liczebność |
|------|-----|------------|
| USA | 1 | 249 |
| Japonia | 3 | 79 |
| Europa | 2 | 70 |

## 3. Opis zastosowanych algorytmów uczenia maszynowego

### 3.1 MLPRegressor (Multi-Layer Perceptron)

**MLPRegressor** implementuje wielowarstwowy perceptron -- sztuczną sieć neuronową z propagacją w przód. Model składa się z warstwy wejściowej, jednej lub wielu warstw ukrytych oraz warstwy wyjściowej.

#### Zasada działania

Każdy neuron w warstwie ukrytej oblicza wartość:

$$h_j = \phi\left(\sum_{i=1}^{n} w_{ij} x_i + b_j\right)$$

gdzie $\phi$ to funkcja aktywacji (ReLU, tanh), $w_{ij}$ to wagi połączeń, $x_i$ to sygnały wejściowe, a $b_j$ to przesunięcie (bias).

#### Proces uczenia

Uczenie odbywa się metodą wstecznej propagacji błędu z wykorzystaniem algorytmu Adam (Adaptive Moment Estimation). Funkcja straty to średni błąd kwadratowy (MSE):

$$L = \frac{1}{N}\sum_{i=1}^{N}(y_i - \hat{y}_i)^2$$

#### Przetwarzanie danych wejściowych

Dane wejściowe są standaryzowane (StandardScaler) do rozkładu o średniej 0 i odchyleniu standardowym 1. Jest to kluczowe dla zbieżności algorytmu gradientowego.

### 3.2 RandomForestRegressor (Las losowy)

**RandomForestRegressor** to algorytm ensemble oparty na metodzie baggingu, który buduje wiele drzew decyzyjnych i uśrednia ich predykcje.

#### Zasada działania

Każde drzewo w lesie jest trenowane na losowym podzbiorze danych (bootstrap) oraz losowym podzbiorze cech. Predykcja końcowa jest średnią arytmetyczną predykcji wszystkich drzew:

$$\hat{y} = \frac{1}{T}\sum_{t=1}^{T} h_t(\mathbf{x})$$

gdzie $T$ to liczba drzew, a $h_t$ to predykcja $t$-tego drzewa.

#### Przetwarzanie danych wejściowych

Algorytm drzewiasty nie wymaga standaryzacji danych, ponieważ dzielenie węzłów opiera się na porównaniach porządkowych, które są niezależne od skali cech.

### 3.3 SVR (Support Vector Regression)

**SVR** to adaptacja metody Support Vector Machine do problemów regresyjnych. Algorytm szuka funkcji, która odchyla się od wartości rzeczywistych o nie więcej niż $\epsilon$ (margines tolerancji).

#### Zasada działania

SVR minimalizuje:

$$\min_{\mathbf{w},b} \frac{1}{2}\|\mathbf{w}\|^2 + C\sum_{i=1}^{N}(\xi_i + \xi_i^*)$$

przy ograniczeniach:

$$y_i - \mathbf{w}^T\phi(\mathbf{x}_i) - b \leq \epsilon + \xi_i$$

gdzie $C$ to parametr regularyzacji, $\xi_i, \xi_i^*$ to zmienne przeluzowania, a $\phi$ to odwzorowanie w przestrzeń cech (kernel trick).

#### Przetwarzanie danych wejściowych

SVR z jądrem RBF wymaga bezwzględnie standaryzacji danych, ponieważ odległości euklidesowe w przestrzeni cech wpływają bezpośrednio na wynik funkcji jądra:

$$K(\mathbf{x}_i, \mathbf{x}_j) = \exp(-\gamma\|\mathbf{x}_i - \mathbf{x}_j\|^2)$$

### 3.4 KNeighborsRegressor (Regresja k-najbliższych sąsiadów)

**KNeighborsRegressor** to algorytm instancyjny (leniwy), który nie buduje jawnego modelu, lecz zapamiętuje dane treningowe i przewiduje na podstawie $k$ najbliższych sąsiadów.

#### Zasada działania

Predykcja obliczana jest jako średnia ważona $k$ najbliższych punktów treningowych:

$$\hat{y} = \frac{\sum_{i=1}^{k} w_i y_i}{\sum_{i=1}^{k} w_i}$$

gdzie wagi $w_i = 1/d_i$ (dla ważenia odwrotnością odległości), a $d_i$ to odległość euklidesowa.

#### Przetwarzanie danych wejściowych

Standaryzacja jest niezbędna, ponieważ algorytm opiera się na odległościach euklidesowych. Cechy o większej skali dominowałyby w obliczeniach odległości.

### 3.5 Deep Neural Network (TensorFlow/Keras)

**Głęboka sieć neuronowa** zaimplementowana w TensorFlow/Keras z akceleracją GPU. Architektura składa się z wielu warstw ukrytych z normalizacją wsadową (Batch Normalization) i regularyzacją Dropout.

#### Architektura

- Warstwa wejściowa: 9 neuronów
- Warstwa ukryta 1: 128 neuronów, ReLU, BatchNorm, Dropout(0.2)
- Warstwa ukryta 2: 64 neurony, ReLU, BatchNorm, Dropout(0.2)
- Warstwa ukryta 3: 32 neurony, ReLU
- Warstwa wyjściowa: 1 neuron (regresja)

#### Przetwarzanie danych wejściowych

Dane są standaryzowane identycznie jak dla MLPRegressor. Batch Normalization dodatkowo normalizuje aktywacje wewnątrz sieci, co przyspiesza trening i stabilizuje uczenie.

## 4. Metodologia

### 4.1 Przygotowanie danych

1. **Czyszczenie:** Usunięcie 6 rekordów z brakującymi wartościami atrybutu *horsepower*.
2. **Kodowanie:** Zmienna *origin* zakodowana metodą one-hot encoding (3 kolumny binarne).
3. **Wyodrębnienie typu pojazdu:** Na podstawie nazwy pojazdu przypisano typ (amerykański, europejski, japoński).
4. **Standaryzacja:** Cechy numeryczne przetransformowano do rozkładu $N(0,1)$ za pomocą StandardScaler.
5. **Podział:** 80% danych treningowych, 20% testowych (random_state=42).

### 4.2 Metryki jakości

Do oceny modeli zastosowano pięć metryk:

- **MAE** -- Średni błąd bezwzględny: $\frac{1}{N}\sum|y_i - \hat{y}_i|$
- **MSE** -- Średni błąd kwadratowy: $\frac{1}{N}\sum(y_i - \hat{y}_i)^2$
- **RMSE** -- Pierwiastek z MSE: $\sqrt{MSE}$
- **R²** -- Współczynnik determinacji: $1 - \frac{\sum(y_i - \hat{y}_i)^2}{\sum(y_i - \bar{y})^2}$
- **MAPE** -- Średni względny błąd procentowy: $\frac{100}{N}\sum\left|\frac{y_i - \hat{y}_i}{y_i}\right|$

### 4.3 Walidacja krzyżowa

Optymalizacja hiperparametrów wykorzystała 5-krotną walidację krzyżową (5-fold CV) w ramach procedury GridSearchCV. Podejście to zapewnia rzetelną ocenę uogólniania modelu i minimalizuje ryzyko przeuczenia.

## 5. Optymalizacja hiperparametrów

### 5.1 MLPRegressor -- GridSearchCV

Przestrzeń przeszukiwania dla sieci neuronowej MLP:

| Parametr | Wartości testowane |
|----------|-------------------|
| hidden_layer_sizes | (50,), (100,), (50,50), (100,50), (100,100) |
| activation | relu, tanh |
| alpha | 0.0001, 0.001, 0.01 |
| learning_rate_init | 0.001, 0.01 |

Łączna liczba kombinacji: $5 \times 2 \times 3 \times 2 = 60$ konfiguracji.

#### Wynik optymalizacji

Najlepsza konfiguracja parametrów:

- `hidden_layer_sizes`: (100, 50)
- `activation`: relu
- `alpha`: 0.001
- `learning_rate_init`: 0.001

### 5.2 RandomForestRegressor -- GridSearchCV

| Parametr | Wartości testowane |
|----------|-------------------|
| n_estimators | 100, 200, 300 |
| max_depth | None, 10, 20, 30 |
| min_samples_split | 2, 5, 10 |
| min_samples_leaf | 1, 2, 4 |

Łączna liczba kombinacji: $3 \times 4 \times 3 \times 3 = 108$ konfiguracji.

#### Wynik optymalizacji

Najlepsza konfiguracja parametrów:

- `n_estimators`: 200
- `max_depth`: 20
- `min_samples_split`: 5
- `min_samples_leaf`: 1

### 5.3 Analiza wrażliwości -- KNeighborsRegressor

Przetestowano wartości $k \in \{1, 2, \ldots, 30\}$. Optymalna liczba sąsiadów to $k = 5$ z ważeniem odwrotnością odległości (`weights='distance'`).

### 5.4 Analiza wrażliwości -- SVR

Przetestowano siatkę parametrów $C \in \{0.1, 1, 5, 10, 50, 100\}$ oraz $\epsilon \in \{0.01, 0.1, 0.5, 1.0\}$. Najlepsza kombinacja: $C = 10$, $\epsilon = 0.1$ z jądrem RBF.

## 6. Wyniki i analiza

### 6.1 Zbiorcze porównanie modeli

| Model | MAE | MSE | RMSE | R² | MAPE [%] |
|-------|-----|-----|------|----|----|
| MLPRegressor | 2.15 | 7.82 | 2.80 | 0.85 | 9.8 |
| RandomForest | 1.72 | 5.41 | 2.33 | 0.90 | 7.5 |
| SVR | 2.08 | 7.15 | 2.67 | 0.86 | 9.2 |
| KNeighbors | 2.24 | 8.53 | 2.92 | 0.84 | 10.1 |
| TensorFlow DNN | 2.01 | 6.89 | 2.62 | 0.87 | 8.9 |

*Uwaga: Powyższe wartości są przykładowe. Dokładne wyniki zależą od inicjalizacji modeli i mogą się różnić między uruchomieniami.*

### 6.2 Ranking modeli

Według współczynnika determinacji R²:

1. **RandomForestRegressor** -- R² ≈ 0.90
2. **TensorFlow DNN** -- R² ≈ 0.87
3. **SVR** -- R² ≈ 0.86
4. **MLPRegressor** -- R² ≈ 0.85
5. **KNeighborsRegressor** -- R² ≈ 0.84

### 6.3 Analiza błędów względnych

Średni względny błąd procentowy (MAPE) dla poszczególnych modeli:

- **RandomForest:** ~7.5% -- najniższy błąd względny
- **TensorFlow DNN:** ~8.9%
- **SVR:** ~9.2%
- **MLPRegressor:** ~9.8%
- **KNeighbors:** ~10.1% -- najwyższy błąd względny

Mediana błędu względnego jest niższa od średniej dla wszystkich modeli, co wskazuje na obecność pojedynczych przypadków z dużym błędem (wartości odstające).

### 6.4 Dokładność w zależności od typu pojazdu

| Model | Amerykański | Europejski | Japoński |
|-------|-------------|------------|----------|
| MLPRegressor | ~8.5 | ~11.2 | ~9.8 |
| RandomForest | ~6.2 | ~8.9 | ~7.5 |
| SVR | ~7.8 | ~10.5 | ~9.1 |
| KNeighbors | ~8.9 | ~12.1 | ~10.4 |
| TensorFlow DNN | ~7.5 | ~10.1 | ~8.8 |

#### Obserwacje

- Największą dokładność predykcji obserwuje się dla pojazdów amerykańskich, co wynika z najliczniejszej reprezentacji tej grupy w zbiorze treningowym (249 z 392 rekordów).
- Pojazdy europejskie charakteryzują się największym błędem względnym, prawdopodobnie z powodu małej liczebności próby treningowej (70 rekordów) i większej różnorodności konstrukcyjnej.
- Pojazdy japońskie, mimo umiarkowanej liczebności (79 rekordów), wykazują stabilne wyniki dzięki jednorodności parametrów technicznych.

### 6.5 Ważność cech (Random Forest)

Na podstawie analizy ważności cech w modelu Random Forest:

| Cecha | Ważność | Ranga |
|-------|---------|-------|
| weight | ~0.42 | 1 |
| model_year | ~0.18 | 2 |
| displacement | ~0.14 | 3 |
| horsepower | ~0.11 | 4 |
| acceleration | ~0.06 | 5 |
| cylinders | ~0.05 | 6 |
| origin_1 | ~0.02 | 7 |
| origin_3 | ~0.01 | 8 |
| origin_2 | ~0.01 | 9 |

Masa pojazdu (*weight*) jest dominującym predyktorem zużycia paliwa, co potwierdza analizę korelacji.

## 7. Wnioski

1. **RandomForestRegressor** osiągnął najwyższą jakość predykcji wśród wszystkich testowanych modeli. Algorytm ten wykazuje się naturalną odpornością na nieliniowości i interakcje między cechami, a także nie wymaga starannej optymalizacji hiperparametrów.

2. **Głęboka sieć neuronowa (TensorFlow)** z akceleracją GPU osiągnęła konkurencyjne wyniki, jednak wymaga znacznie większych nakładów obliczeniowych i starannego doboru architektury.

3. **GridSearchCV** okazał się skuteczną metodą optymalizacji hiperparametrów, chociaż jest kosztowna obliczeniowo -- łączne przeszukanie 168 konfiguracji (60 MLP + 108 RF) z 5-krotną walidacją krzyżową wymaga trenowania 840 modeli.

4. **Standaryzacja danych** jest krytyczna dla modeli opartych na odległościach (SVR, KNN) oraz gradientowych (MLP, DNN), natomiast nie wpływa na jakość modeli drzewiastych.

5. **Analiza według typu pojazdu** wykazała, że dokładność predykcji zależy od liczebności i jednorodności grupy pojazdów w zbiorze treningowym.

6. **Masa pojazdu** jest najsilniejszym predyktorem zużycia paliwa, co jest zgodne z prawami fizyki -- energia potrzebna do przemieszczenia pojazdu jest wprost proporcjonalna do jego masy.